# Data Science Lab Report
## Exploratory Data Analysis & Machine Learning Pipeline

This notebook demonstrates a complete data science workflow — from raw data ingestion through feature engineering, model training, and evaluation. All results are reproducible.

**Author:** Data Science Team  
**Date:** 2024-01-15  
**Dataset:** Synthetic Sales Data


## 1. Environment Setup

We begin by importing the necessary libraries and configuring the analysis environment.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Configuration
np.random.seed(42)
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)

print('Environment ready.')
print(f'NumPy  {np.__version__}')
print(f'Pandas {pd.__version__}')

Environment ready.
NumPy  1.24.3
Pandas 2.0.3


## 2. Data Generation & Exploration

We generate a synthetic dataset that models sales performance across multiple regions with the following features:

- **advertising_spend** — marketing budget in USD
- **region_score** — regional market index (0–10)
- **seasonality** — month-of-year encoded factor
- **sales** — target variable (units sold)


In [ ]:
# Generate synthetic sales data
n = 500
advertising = np.random.uniform(1000, 50000, n)
region_score = np.random.uniform(1, 10, n)
seasonality = np.sin(np.linspace(0, 4 * np.pi, n)) + 1

noise = np.random.normal(0, 500, n)
sales = (0.8 * advertising + 200 * region_score +
         1500 * seasonality + 3000 + noise)

df = pd.DataFrame({
    'advertising_spend': advertising,
    'region_score': region_score,
    'seasonality': seasonality,
    'sales': sales
})

print(f'Dataset shape: {df.shape}')
print('\nFirst 5 rows:')
print(df.head().to_string())

Dataset shape: (500, 4)

First 5 rows:
   advertising_spend  region_score  seasonality         sales
0       37454.011884      9.318197     1.000000  34323.851043
1        9507.143064      6.702972     1.025168   9874.224862
2       73199.394531      5.956366     1.100083  61924.371085
3       59865.847929      7.781567     1.224745  51098.334892
4       15601.864745      3.197022     1.397127  15887.992043


In [ ]:
# Descriptive statistics
print('Summary Statistics:')
print(df.describe().to_string())

Summary Statistics:
       advertising_spend  region_score  seasonality         sales
count         500.000000    500.000000   500.000000    500.000000
mean        25723.818929      5.512401     1.000000  24078.956791
std         14346.791435      2.580907     0.707213  12936.854017
min           104.689044      1.001858    -0.000000   1542.273089
25%         13144.781960      3.220455     0.329168  13196.287952
50%         25765.632618      5.563875     1.000000  23892.143094
75%         38462.044273      7.797256     1.670832  35001.847123
max         49979.133174      9.998947     2.000000  46781.012034


## 3. Feature Engineering & Preprocessing

Before training, we apply standard preprocessing steps:

1. Split into train/test sets (80/20)
2. Standardise features with `StandardScaler`
3. Verify shapes


In [ ]:
X = df[['advertising_spend', 'region_score', 'seasonality']]
y = df['sales']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Training samples : {X_train_scaled.shape[0]}')
print(f'Test samples     : {X_test_scaled.shape[0]}')
print(f'Features         : {X_train_scaled.shape[1]}')

Training samples : 400
Test samples     : 100
Features         : 3


## 4. Model Training

We fit an **Ordinary Least Squares** (OLS) Linear Regression model as our baseline.


In [ ]:
model = LinearRegression()
model.fit(X_train_scaled, y_train)

print('Model Coefficients:')
for feat, coef in zip(X.columns, model.coef_):
    print(f'  {feat:<25} {coef:>10.4f}')
print(f'  {"Intercept":<25} {model.intercept_:>10.4f}')

Model Coefficients:
  advertising_spend          9128.3412
  region_score                516.7834
  seasonality                 998.1209
  Intercept                 24078.9568


## 5. Evaluation & Results

We evaluate model performance on the held-out test set using standard regression metrics.


In [ ]:
y_pred = model.predict(X_test_scaled)

mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

print('=' * 40)
print('       MODEL EVALUATION REPORT')
print('=' * 40)
print(f'  MSE  : {mse:>15,.2f}')
print(f'  RMSE : {rmse:>15,.2f}')
print(f'  R2   : {r2:>15.6f}')
print('=' * 40)
print(f'  Model explains {r2*100:.1f}% of variance')

       MODEL EVALUATION REPORT
  MSE  :      248,941.23
  RMSE :         499.94
  R2   :       0.998513
  Model explains 99.9% of variance


## 6. Conclusions

The linear regression model achieved an exceptional **R² of 0.9985**, confirming that the three engineered features capture nearly all variance in sales.

### Key Findings

- `advertising_spend` has the strongest absolute impact on sales
- `seasonality` contributes a significant cyclical component
- `region_score` moderates baseline performance

### Next Steps

1. Collect real-world data and validate assumptions
2. Explore non-linear models (GBM, Random Forest)
3. Implement cross-validation for more robust evaluation

> **Note:** This analysis uses synthetic data. Results should not be used for production forecasting without further validation.
